In [2]:
import os
import multiprocessing as mp
import tiktoken
from datasets import load_dataset, Dataset, concatenate_datasets
from tqdm import tqdm  
from pathlib import Path
import torch
import numpy as np
from itertools import accumulate

import sys
sys.path.append("../src")
import settings as s
from dataset import ShakespearDataset, load_tokens

In [29]:
data_path = Path("../data/edu_fineweb1B/shards")
npy_files = list(data_path.glob("fineweb-edu-train-*.npy"))
npy_files

[WindowsPath('../data/edu_fineweb1B/shards/fineweb-edu-train-000001.npy'),
 WindowsPath('../data/edu_fineweb1B/shards/fineweb-edu-train-000002.npy'),
 WindowsPath('../data/edu_fineweb1B/shards/fineweb-edu-train-000003.npy'),
 WindowsPath('../data/edu_fineweb1B/shards/fineweb-edu-train-000004.npy'),
 WindowsPath('../data/edu_fineweb1B/shards/fineweb-edu-train-000005.npy'),
 WindowsPath('../data/edu_fineweb1B/shards/fineweb-edu-train-000006.npy'),
 WindowsPath('../data/edu_fineweb1B/shards/fineweb-edu-train-000007.npy'),
 WindowsPath('../data/edu_fineweb1B/shards/fineweb-edu-train-000008.npy'),
 WindowsPath('../data/edu_fineweb1B/shards/fineweb-edu-train-000009.npy'),
 WindowsPath('../data/edu_fineweb1B/shards/fineweb-edu-train-000010.npy')]

In [2]:
remote_name = "sample-10BT"
shard_size = int(1e8)  # 100M tokens per shard, total of 10 shards

shards_dir = s.data_path/"shards"
shards_dir.mkdir(parents=True, exist_ok=True)

dataset = load_dataset(
    path="HuggingFaceFW/fineweb-edu",
    name=remote_name,
    split="train",
    cache_dir=s.data_path,
)
dataset

Resolving data files:   0%|          | 0/2110 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/98 [00:00<?, ?it/s]

Dataset({
    features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score'],
    num_rows: 9672101
})

In [3]:
dataset[0]

{'text': 'The Independent Jane\nFor all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.\nElizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.\nThe freedom she exhibited in finally accepting him in direct defiance of Lady Catherine and knowing her father would disapprove was unusual even for Austen. In her last book Anne Elliot is persuaded to refuse Captain Wentworth at Lady Russel’s insistence.\nAlthough Jane played by the rules of the day, all of her writing is infused with how she wanted life to be. She ‘screams’ her outrage at the limitations for women in Emma.\nWhen accosted by Mrs. Elton, Jane Fairfax says,\n“Excuse me, ma’am, but this is by no means my intention; I make no inquiry myself, and sho

In [4]:
dataset[:2]["text"]

['The Independent Jane\nFor all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.\nElizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.\nThe freedom she exhibited in finally accepting him in direct defiance of Lady Catherine and knowing her father would disapprove was unusual even for Austen. In her last book Anne Elliot is persuaded to refuse Captain Wentworth at Lady Russel’s insistence.\nAlthough Jane played by the rules of the day, all of her writing is infused with how she wanted life to be. She ‘screams’ her outrage at the limitations for women in Emma.\nWhen accosted by Mrs. Elton, Jane Fairfax says,\n“Excuse me, ma’am, but this is by no means my intention; I make no inquiry myself, and should be s

In [10]:
# init the tokenizer
enc = tiktoken.get_encoding("gpt2")
eot = enc._special_tokens['<|endoftext|>']  # end of text token

def tokenize(doc):
    # tokenizes a single document and returns a numpy array of uint16 tokens
    tokens = [eot]  # the special <|endoftext|> token delimits all documents
    tokens.extend(enc.encode_ordinary(doc["text"]))
    tokens_np = np.array(tokens)
    assert (0 <= tokens_np).all() and (tokens_np < 2**16).all(), "token dictionary too large for uint16"
    tokens_np_uint16 = tokens_np.astype(np.uint16)

    return tokens_np_uint16


def write_datafile(filename, tokens_np):
    np.save(filename, tokens_np)

# tokenize all documents and write output shards, each of shard_size tokens (last shard has remainder)
nprocs = max(1, os.cpu_count()//2)
nprocs

4

In [ ]:
shard_size = int(1e8)  # 100M tokens per shard, total of 10 shards

with mp.Pool(nprocs) as pool:
    shard_index = 0
    # preallocate buffer to hold current shard
    all_tokens_np = np.empty((shard_size,), dtype=np.uint16)
    token_count = 0
    progress_bar = None
    for tokens in pool.imap(tokenize, dataset, chunksize=16):
        # is there enough space in the current shard for the new tokens?
        if token_count + len(tokens) < shard_size:
            # simply append tokens to current shard
            all_tokens_np[token_count:token_count+len(tokens)] = tokens
            token_count += len(tokens)
            # update progress bar
            if progress_bar is None:
                progress_bar = tqdm(
                    total=shard_size, unit="tokens", desc=f"Shard {shard_index}")
            progress_bar.update(len(tokens))
        else:
            # write the current shard and start a new one
            split = "val" if shard_index == 0 else "train"
            filename = shards_dir / f"fineweb-edu-{split}-{shard_index:06d}"
            # split the document into whatever fits in this shard; the remainder goes to next one
            remainder = shard_size - token_count
            progress_bar.update(remainder)
            all_tokens_np[token_count:token_count +
                          remainder] = tokens[:remainder]
            write_datafile(filename, all_tokens_np)
            shard_index += 1
            progress_bar = None
            # populate the next shard with the leftovers of the current doc
            all_tokens_np[0:len(tokens)-remainder] = tokens[remainder:]
            token_count = len(tokens)-remainder

    # write any remaining tokens as the last shard
    if token_count != 0:
        split = "val" if shard_index == 0 else "train"
        filename = shards_dir / f"fineweb-edu-{split}-{shard_index:06d}"
        write_datafile(filename, all_tokens_np[:token_count])


In [5]:
arrow_files = list(s.data_path.rglob("fineweb-edu-train-*.arrow"))
datasets = [Dataset.from_file(str(f)) for f in arrow_files[:12]]

from datasets import concatenate_datasets
small_dataset = concatenate_datasets(datasets)
small_dataset

Dataset({
    features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score'],
    num_rows: 1186000
})

In [6]:
dataset = small_dataset
data_path = Path("../data/edu_fineweb100M")
shards_dir = data_path/"shards"
shards_dir.mkdir(parents=True, exist_ok=True)

In [8]:

# Initialize tiktoken tokenizer
tokenizer = tiktoken.get_encoding("gpt2")
eot = tokenizer._special_tokens['<|endoftext|>']  # end of text token

# Function to count tokens in each example
def count_tokens(example):
    tokens = [eot]  # the special <|endoftext|> token delimits all documents
    tokens.extend(tokenizer.encode_ordinary(example["text"]))
    
    return {"token_count": len(tokens)}

# Add token counts to the dataset
dataset_with_tokens = dataset.map(count_tokens, desc="Counting tokens")

# Compute cumulative token counts
token_counts = dataset_with_tokens["token_count"]
cumulative_tokens = list(accumulate(token_counts))

# Find cutoff index to reach 1B tokens
token_limit = int(1e9)
cutoff_idx = next(idx for idx, total in enumerate(cumulative_tokens) if total >= token_limit)

# Select the subset of examples
subset_dataset = dataset_with_tokens.select(range(cutoff_idx + 1))
subset_dataset


Counting tokens:   0%|          | 0/1186000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score'],
    num_rows: 968516
})

In [9]:
dataset = subset_dataset 

In [1]:
subset_dataset.save_to_disk("1B_tokens")

NameError: name 'subset_dataset' is not defined